# Semi-Supervised Graph Learning & Active Learning Visualizations

Welcome to the demonstration suite for the `graphalp` package. This notebook contains the implementation to generate representative graphs, execute label propagation and active learning samplers, and visualize their step-by-step convergence as animated GIFs.

We use a **20x20 Grid Graph** with a **100-node central island of 1s** in a background of 0s. This beautifully demonstrates how active learning samplers locate structured island boundaries.

### Visualized Algorithms
1. **Label Propagation**
   * **Harmonic LP (Dirichlet Problem)**: Jacobi iterative convergence showing probability diffusion.
   * **Min-Cut LP**: Network flow source-sink cuts separating classes.
   * **Spectral LP**: Dimensionality reduction using Laplacian eigenvectors + SVM boundaries.
2. **Active Learning Samplers**
   * **Random Sampler**: Uniform baseline queries.
   * **S2 Sampler**: Path bisection search along labeled boundaries.
   * **Harmonic Greedy Sampler**: Expected risk minimization querying the most uncertain nodes.

In [ ]:
import sys
import os
import io
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

# Ensure local src/ folder is imported correctly
sys.path.insert(0, os.path.abspath('../src'))

from graphalp.label_propagation import HarmonicLabelPropagator, MinCutLabelPropagator, SpectralLabelPropagator
from graphalp.active_learning import HarmonicGreedySampler, S2Sampler, RandomSampler
from graphalp.utils import compute_risk

## 1. Setup Theme & Grid Island Graph Generation

We define a custom theme with a white background and solid black borders around queried/labeled nodes on a 20x20 Grid.

In [ ]:
# Custom white background visuals
BG_COLOR = '#ffffff'
EDGE_COLOR = '#94a3b8'
NODE_BLUE = '#2563eb'
BORDER_BLUE = '#000000'
NODE_RED = '#e11d48'
BORDER_RED = '#000000'
NODE_GREY = '#cbd5e1'
BORDER_GREY = '#cbd5e1'
HIGHLIGHT_GOLD = '#d97706'
TEXT_COLOR = '#0f172a'

def get_predicted_color(p):
    r = int(37 + (225 - 37) * p)
    g = int(99 + (29 - 99) * p)
    b = int(235 + (72 - 235) * p)
    return f'#{r:02x}{g:02x}{b:02x}'

def fig_to_img(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', facecolor=fig.get_facecolor(), edgecolor='none', dpi=100)
    buf.seek(0)
    img = Image.open(buf)
    img.load()
    buf.close()
    return img

def draw_base_graph(ax, G, pos, labels, predictions, highlight_node=None, title=""):
    ax.set_facecolor(BG_COLOR)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=EDGE_COLOR, width=0.8, alpha=0.3)
    for node in G.nodes():
        label = labels.get(node)
        pred = predictions.get(node, 0.5)
        if label == 0:
            color, border, width, size = NODE_BLUE, BORDER_BLUE, 2.0, 130
        elif label == 1:
            color, border, width, size = NODE_RED, BORDER_RED, 2.0, 130
        else:
            color, border, width, size = get_predicted_color(pred), BORDER_GREY, 0.5, 55
        nx.draw_networkx_nodes(G, pos, nodelist=[node], node_color=color, node_size=size,
                               edgecolors=border, linewidths=width, ax=ax)
    if highlight_node is not None:
        nx.draw_networkx_nodes(G, pos, nodelist=[highlight_node], node_color='none', node_size=250,
                               edgecolors=HIGHLIGHT_GOLD, linewidths=2.5, ax=ax)
        x, y = pos[highlight_node]
        ax.text(x, y + 1.2, "Query!", color=HIGHLIGHT_GOLD, fontsize=9, fontweight='bold',
                horizontalalignment='center', bbox=dict(facecolor=BG_COLOR, edgecolor=HIGHLIGHT_GOLD, boxstyle='round,pad=0.15'))
    ax.set_title(title, color=TEXT_COLOR, fontsize=13, fontweight='bold', pad=10)
    ax.axis('off')

# 20x20 Grid Generation
G = nx.grid_2d_graph(20, 20)
mapping = {}
for y in range(20):
    for x in range(20):
        mapping[(y, x)] = y * 20 + x
G = nx.relabel_nodes(G, mapping)

pos = {}
for y in range(20):
    for x in range(20):
        pos[y * 20 + x] = np.array([float(x), float(y)])

gt_labels = {}
for n in G.nodes():
    x_coord = n % 20
    y_coord = n // 20
    if 5 <= x_coord <= 14 and 5 <= y_coord <= 14:
        gt_labels[n] = 1
    else:
        gt_labels[n] = 0

fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, {0:0, 189:1}, {}, title="Starting Grid Island Graph Structure")
plt.show()

## 2. Harmonic Label Propagation - Jacobi Iteration Animation

We show the Jacobi solver running step-by-step, allowing us to watch label probabilities diffuse organically outwards from the seeds.

In [ ]:
print("Generating Harmonic LP...")
frames = []
labels = {0: 0, 189: 1}
f_iter = {n: 0.5 for n in G.nodes()}
f_iter[0] = 0.0
f_iter[189] = 1.0

for step in range(21):
    fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
    title = f"Harmonic Label Propagation (Jacobi Iteration {step})"
    draw_base_graph(ax, G, pos, labels, f_iter, title=title)
    plt.tight_layout()
    frames.append(fig_to_img(fig))
    plt.close(fig)
    
    f_next = f_iter.copy()
    for u in G.nodes():
        if u not in labels:
            neighbors = list(G.neighbors(u))
            f_next[u] = sum(f_iter[v] for v in neighbors) / len(neighbors)
    f_iter = f_next

os.makedirs('../docs/gifs', exist_ok=True)
frames += [frames[-1]] * 15
frames[0].save('../docs/gifs/harmonic_propagation.gif', save_all=True, append_images=frames[1:], duration=350, loop=0)
print("Saved docs/gifs/harmonic_propagation.gif")

## 3. Min-Cut Label Propagation Animation

We show the minimum cut algorithm adding source/sink edges, discovering the mincut wall, and creating hard bipartitions.

In [ ]:
print("Generating Min-Cut LP...")
frames = []
labels = {0: 0, 189: 1}

# Frame 0: Initial
fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, labels, {}, title="Min-Cut: Starting Seed Labels")
plt.tight_layout()
f0 = fig_to_img(fig)
plt.close(fig)
frames += [f0] * 12

# Frame 1: Source & Sink
fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, labels, {}, title="Min-Cut: Connect to Virtual Source & Sink")
src_pos, snk_pos = (-2.5, 9.5), (21.5, 9.5)
ax.scatter(src_pos[0], src_pos[1], color='#10b981', s=250, marker='s', edgecolors='#047857', zorder=5)
ax.text(src_pos[0], src_pos[1]-1.2, "Source (s)", color='#10b981', fontweight='bold', horizontalalignment='center')
ax.scatter(snk_pos[0], snk_pos[1], color='#8b5cf6', s=250, marker='s', edgecolors='#6d28d9', zorder=5)
ax.text(snk_pos[0], snk_pos[1]-1.2, "Sink (t)", color='#8b5cf6', fontweight='bold', horizontalalignment='center')
ax.plot([src_pos[0], pos[0][0]], [src_pos[1], pos[0][1]], color='#10b981', linestyle='--', linewidth=1.5)
ax.plot([snk_pos[0], pos[189][0]], [snk_pos[1], pos[189][1]], color='#8b5cf6', linestyle='--', linewidth=1.5)
plt.tight_layout()
f1 = fig_to_img(fig)
plt.close(fig)
frames += [f1] * 12

# Frame 2: MinCut Wall
propagator = MinCutLabelPropagator(G)
propagator.fit([0, 189], [0, 1])
cut_edges = propagator.mincut_wall
fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, labels, {}, title="Min-Cut: Compute Minimum s-t Cut")
nx.draw_networkx_edges(G, pos, ax=ax, edgelist=cut_edges, edge_color=HIGHLIGHT_GOLD, width=2.5, style='dashed')
plt.tight_layout()
f2 = fig_to_img(fig)
plt.close(fig)
frames += [f2] * 15

# Frame 3: Final Partition
predictions = {n: 1 if n in propagator.reachable else 0 for n in G.nodes()}
fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, labels, predictions, title="Min-Cut: Final Hard Class Bipartition")
nx.draw_networkx_edges(G, pos, ax=ax, edgelist=cut_edges, edge_color=HIGHLIGHT_GOLD, width=1.5, style='dotted', alpha=0.6)
plt.tight_layout()
f3 = fig_to_img(fig)
plt.close(fig)
frames += [f3] * 20

frames[0].save('../docs/gifs/mincut_propagation.gif', save_all=True, append_images=frames[1:], duration=150, loop=0)
print("Saved docs/gifs/mincut_propagation.gif")

## 4. Spectral Label Propagation Animation

We show the mapping of nodes to the 2D Laplacian eigenvector space, the fit of the SVM decision boundary, and the back-projected probabilities.

In [ ]:
print("Generating Spectral LP...")
frames = []
labels = {0: 0, 189: 1}

# Frame 0: Physical
fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, labels, {}, title="Spectral LP: Physical Graph")
plt.tight_layout()
s0 = fig_to_img(fig)
plt.close(fig)
frames += [s0] * 12

# Compute Spectral
spectral_prop = SpectralLabelPropagator(G, n_components=3)
spectral_prop.fit([0, 189], [0, 1])
embeddings = spectral_prop.embeddings[:, 1:3]

# Frame 1: Embedding Space Scatter
fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
ax.set_facecolor(BG_COLOR)
ax.spines['bottom'].set_color(EDGE_COLOR)
ax.spines['left'].set_color(EDGE_COLOR)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(colors=TEXT_COLOR)
for n in G.nodes():
    emb = embeddings[n]
    color, border, size = (NODE_BLUE, BORDER_BLUE, 150) if n==0 else ((NODE_RED, BORDER_RED, 150) if n==189 else (NODE_GREY, BORDER_GREY, 40))
    ax.scatter(emb[0], emb[1], color=color, s=size, edgecolors=border, zorder=5)
ax.set_title("Spectral LP: Nodes in 2D Eigenvector Space", color=TEXT_COLOR, fontsize=13, fontweight='bold', pad=10)
ax.set_xlabel("Eigenvector 1", color=TEXT_COLOR)
ax.set_ylabel("Eigenvector 2", color=TEXT_COLOR)
plt.tight_layout()
s1 = fig_to_img(fig)
plt.close(fig)
frames += [s1] * 12

# Frame 2: SVM Boundaries
fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
ax.set_facecolor(BG_COLOR)
ax.spines['bottom'].set_color(EDGE_COLOR)
ax.spines['left'].set_color(EDGE_COLOR)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(colors=TEXT_COLOR)

x_min, x_max = embeddings[:, 0].min() - 0.02, embeddings[:, 0].max() + 0.02
y_min, y_max = embeddings[:, 1].min() - 0.02, embeddings[:, 1].max() + 0.02
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid_points = np.c_[xx.ravel(), yy.ravel()]

const_val = spectral_prop.embeddings[0, 0]
grid_points_3d = np.c_[np.full(grid_points.shape[0], const_val), grid_points]
Z = spectral_prop.svc.predict_proba(grid_points_3d)[:, 1].reshape(xx.shape)

colors = [(37/255, 99/255, 235/255), (1.0, 1.0, 1.0), (225/255, 29/255, 72/255)]
cmap_spectral = LinearSegmentedColormap.from_list('blue_red_spectral_white', colors, N=100)

ax.contourf(xx, yy, Z, levels=50, cmap=cmap_spectral, alpha=0.3, zorder=1)
ax.contour(xx, yy, Z, levels=[0.5], colors=HIGHLIGHT_GOLD, linewidths=2.0, linestyles='dashed', zorder=2)
for n in G.nodes():
    emb = embeddings[n]
    color, border, size = (NODE_BLUE, BORDER_BLUE, 150) if n==0 else ((NODE_RED, BORDER_RED, 150) if n==189 else (NODE_GREY, BORDER_GREY, 40))
    ax.scatter(emb[0], emb[1], color=color, s=size, edgecolors=border, zorder=5)
ax.set_title("Spectral LP: Fit SVM in Embedding Space", color=TEXT_COLOR, fontsize=13, fontweight='bold', pad=10)
ax.set_xlabel("Eigenvector 1", color=TEXT_COLOR)
ax.set_ylabel("Eigenvector 2", color=TEXT_COLOR)
plt.tight_layout()
s2 = fig_to_img(fig)
plt.close(fig)
frames += [s2] * 15

# Frame 3: Physical predictions
preds_spectral = dict(zip(G.nodes(), spectral_prop.predict_probabilities(list(G.nodes()))))
fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
draw_base_graph(ax, G, pos, labels, preds_spectral, title="Spectral LP: Predicted Probabilities on Graph")
plt.tight_layout()
s3 = fig_to_img(fig)
plt.close(fig)
frames += [s3] * 20

frames[0].save('../docs/gifs/spectral_propagation.gif', save_all=True, append_images=frames[1:], duration=150, loop=0)
print("Saved docs/gifs/spectral_propagation.gif")

## 5. Active Learning Sampler Comparison Loop

We run the Random, S2, and Harmonic Greedy active learning loops for 15 queries on the 20x20 grid, saving visual GIFs and plotting risk reduction and accuracy curves.

In [ ]:
sampler_classes = {
    'harmonic_greedy': (HarmonicGreedySampler, "Harmonic Greedy Sampler"),
    's2': (lambda G: S2Sampler(G, fallback_sampler=RandomSampler(G)), "S2 Sampler"),
    'random': (RandomSampler, "Random Sampler")
}
all_sampler_risks = {}
all_sampler_accuracies = {}

def get_accuracy(pred_dict, gt_dict):
    correct = 0
    for node, p in pred_dict.items():
        pred_label = 1 if p >= 0.5 else 0
        if pred_label == gt_dict[node]:
            correct += 1
    return (correct / len(gt_dict)) * 100

for key, (cls_init, name) in sampler_classes.items():
    print(f"Running {name}...")
    frames = []
    np.random.seed(42)
    X_labeled, y_labeled = [0, 189], [0, 1]
    sampler = cls_init(G)
    sampler.fit(X_labeled, y_labeled)
    
    prop = HarmonicLabelPropagator(G)
    prop.fit(X_labeled, y_labeled)
    all_preds = dict(zip(G.nodes(), prop.predict_probabilities(list(G.nodes()))))
    initial_risk = compute_risk(np.array([prop.f[n] for n in prop.nodes if n not in prop.labels]))
    initial_acc = get_accuracy(all_preds, gt_labels)
    
    risks = [initial_risk]
    accuracies = [initial_acc]
    
    fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
    draw_base_graph(ax, G, pos, dict(zip(X_labeled, y_labeled)), all_preds, title=f"{name}: Initial\n(Risk: {initial_risk:.2f}, Accuracy: {initial_acc:.1f}%)")
    plt.tight_layout()
    frames.append(fig_to_img(fig))
    plt.close(fig)
    
    for step in range(1, 16):
        next_node = sampler.sample()
        if next_node is None: break
        label = gt_labels[next_node]
        
        fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
        draw_base_graph(ax, G, pos, dict(zip(X_labeled, y_labeled)), all_preds, highlight_node=next_node, title=f"{name}: Query Node {next_node} (Step {step})")
        plt.tight_layout()
        frames.append(fig_to_img(fig))
        plt.close(fig)
        
        X_labeled.append(next_node)
        y_labeled.append(label)
        sampler.fit(X_labeled, y_labeled)
        
        prop.fit(X_labeled, y_labeled)
        all_preds = dict(zip(G.nodes(), prop.predict_probabilities(list(G.nodes()))))
        current_risk = compute_risk(np.array([prop.f[n] for n in prop.nodes if n not in prop.labels]))
        current_acc = get_accuracy(all_preds, gt_labels)
        
        risks.append(current_risk)
        accuracies.append(current_acc)
        
        fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=BG_COLOR)
        draw_base_graph(ax, G, pos, dict(zip(X_labeled, y_labeled)), all_preds, title=f"{name}: Post-Query\n(Risk: {current_risk:.2f}, Accuracy: {current_acc:.1f}%)")
        plt.tight_layout()
        frames.append(fig_to_img(fig))
        plt.close(fig)
        
    all_sampler_risks[key] = risks
    all_sampler_accuracies[key] = accuracies
    frames += [frames[-1]] * 10
    frames[0].save(f'../docs/gifs/{key}_sampler.gif', save_all=True, append_images=frames[1:], duration=600, loop=0)
    print(f"Saved docs/gifs/{key}_sampler.gif")

# Dual-Panel Comparison Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG_COLOR)
steps = np.arange(0, 16)

ax1.set_facecolor(BG_COLOR)
ax1.spines['bottom'].set_color(EDGE_COLOR)
ax1.spines['left'].set_color(EDGE_COLOR)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.tick_params(colors=TEXT_COLOR)
ax1.plot(steps, all_sampler_risks['harmonic_greedy'], color='#d97706', marker='o', linewidth=2.5, label='Harmonic Greedy')
ax1.plot(steps, all_sampler_risks['s2'], color='#8b5cf6', marker='s', linewidth=2.0, label='S2 Sampler')
ax1.plot(steps, all_sampler_risks['random'], color='#64748b', marker='^', linewidth=1.5, linestyle='--', label='Random Sampler')
ax1.set_title("Expected Graph Uncertainty (Risk)", color=TEXT_COLOR, fontsize=12, fontweight='bold')
ax1.set_xlabel("Query Step", color=TEXT_COLOR)
ax1.set_ylabel("Graph Risk", color=TEXT_COLOR)
ax1.grid(color='#cbd5e1', linestyle=':', linewidth=0.5)
ax1.legend(facecolor=BG_COLOR, edgecolor=EDGE_COLOR, labelcolor=TEXT_COLOR)

ax2.set_facecolor(BG_COLOR)
ax2.spines['bottom'].set_color(EDGE_COLOR)
ax2.spines['left'].set_color(EDGE_COLOR)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.tick_params(colors=TEXT_COLOR)
ax2.plot(steps, all_sampler_accuracies['harmonic_greedy'], color='#d97706', marker='o', linewidth=2.5, label='Harmonic Greedy')
ax2.plot(steps, all_sampler_accuracies['s2'], color='#8b5cf6', marker='s', linewidth=2.0, label='S2 Sampler')
ax2.plot(steps, all_sampler_accuracies['random'], color='#64748b', marker='^', linewidth=1.5, linestyle='--', label='Random Sampler')
ax2.set_title("Prediction Accuracy vs. Actual Ground Truth", color=TEXT_COLOR, fontsize=12, fontweight='bold')
ax2.set_xlabel("Query Step", color=TEXT_COLOR)
ax2.set_ylabel("Classification Accuracy (%)", color=TEXT_COLOR)
ax2.grid(color='#cbd5e1', linestyle=':', linewidth=0.5)
ax2.legend(facecolor=BG_COLOR, edgecolor=EDGE_COLOR, labelcolor=TEXT_COLOR)

plt.suptitle("Active Learning Sampler Benchmarks", color=TEXT_COLOR, fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
fig.savefig('../docs/images/risk_comparison.png', facecolor=fig.get_facecolor(), dpi=150)
plt.show()